In [1]:
#!/usr/bin/env python
# coding: utf-8

import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

# Load the dataset
data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv'
)

# Initialize Dash app
app = dash.Dash(__name__)

# List of years
year_list = [i for i in range(1980, 2024)]

# App layout
app.layout = html.Div([
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={'textAlign': 'center', 'color': '#503D36', 'fontSize': 24}
    ),
    
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=[
                {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
                {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
            ],
            value='Yearly Statistics',
            placeholder='Select a report type'
        )
    ], style={'width': '50%', 'margin': '10px'}),
    
    html.Div([
        dcc.Dropdown(
            id='select-year',
            options=[{'label': year, 'value': year} for year in year_list],
            value=1980,
            placeholder='Select Year'
        )
    ], style={'width': '50%', 'margin': '10px'}),
    
    html.Div(id='output-container', style={'display': 'flex', 'flexDirection': 'column'})
])

# Callback to enable/disable year dropdown
@app.callback(
    Output('select-year', 'disabled'),
    Input('dropdown-statistics', 'value')
)
def toggle_year_dropdown(selected_statistics):
    return selected_statistics != 'Yearly Statistics'

# Callback to generate plots
@app.callback(
    Output('output-container', 'children'),
    [Input('dropdown-statistics', 'value'),
     Input('select-year', 'value')]
)
def update_output_container(selected_statistics, input_year):
    if selected_statistics == 'Recession Period Statistics':
        recession_data = data[data['Recession'] == 1]
        
        # Plot 1: Average automobile sales per year
        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(figure=px.line(
            yearly_rec, x='Year', y='Automobile_Sales',
            title="Average Automobile Sales Fluctuation over Recession Period",
            labels={'Automobile_Sales': 'Average Sales', 'Year': 'Year'}
        ))

        # Plot 2: Average sales by vehicle type
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        R_chart2 = dcc.Graph(figure=px.bar(
            average_sales, x='Vehicle_Type', y='Automobile_Sales',
            title="Average Vehicles Sold by Type during Recession",
            labels={'Automobile_Sales': 'Average Sales', 'Vehicle_Type': 'Vehicle Type'}
        ))

        # Plot 3: Advertising expenditure share
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(figure=px.pie(
            exp_rec, names='Vehicle_Type', values='Advertising_Expenditure',
            title="Advertising Expenditure Share by Vehicle Type during Recession"
        ))

        # Plot 4: Effect of unemployment rate
        unemp_data = recession_data.groupby(['Vehicle_Type', 'unemployment_rate'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(figure=px.bar(
            unemp_data, x='unemployment_rate', y='Automobile_Sales', color='Vehicle_Type',
            labels={'unemployment_rate': 'Unemployment Rate', 'Automobile_Sales': 'Average Automobile Sales'},
            title='Effect of Unemployment Rate on Vehicle Type and Sales'
        ))

        return [
            html.Div([html.Div(R_chart1), html.Div(R_chart2)], style={'display': 'flex'}),
            html.Div([html.Div(R_chart3), html.Div(R_chart4)], style={'display': 'flex'})
        ]

    elif selected_statistics == 'Yearly Statistics' and input_year:
        yearly_data = data[data['Year'] == int(input_year)]
        
        # Plot 1: Yearly Automobile Sales (line)
        yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        Y_chart1 = dcc.Graph(figure=px.line(
            yas, x='Year', y='Automobile_Sales',
            title='Yearly Automobile Sales (All Years)',
            labels={'Automobile_Sales': 'Average Sales', 'Year': 'Year'}
        ))

        # Plot 2: Monthly Automobile Sales
        mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
        Y_chart2 = dcc.Graph(figure=px.line(
            mas, x='Month', y='Automobile_Sales',
            title=f'Total Monthly Automobile Sales in {input_year}',
            labels={'Automobile_Sales': 'Total Sales', 'Month': 'Month'}
        ))

        # Plot 3: Average vehicles sold by type
        avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        Y_chart3 = dcc.Graph(figure=px.bar(
            avr_vdata, x='Vehicle_Type', y='Automobile_Sales',
            title=f'Average Vehicles Sold by Vehicle Type in {input_year}',
            labels={'Automobile_Sales': 'Average Sales', 'Vehicle_Type': 'Vehicle Type'}
        ))

        # Plot 4: Advertisement Expenditure by type
        exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        Y_chart4 = dcc.Graph(figure=px.pie(
            exp_data, names='Vehicle_Type', values='Advertising_Expenditure',
            title=f'Total Advertisement Expenditure by Vehicle Type in {input_year}'
        ))

        return [
            html.Div([html.Div(Y_chart1), html.Div(Y_chart2)], style={'display': 'flex'}),
            html.Div([html.Div(Y_chart3), html.Div(Y_chart4)], style={'display': 'flex'})
        ]

    return None

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)


ModuleNotFoundError: No module named 'dash'

In [2]:
pip install dash


   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.9 MB 3.8 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/7.9 MB 5.1 MB/s eta 0:00:02
   ------------- -------------------------- 2.6/7.9 MB 3.8 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/7.9 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 3.7/7.9 MB 3.3 MB/s eta 0:00:02
   --------------------- ------------------ 4.2/7.9 MB 3.3 MB/s eta 0:00:02
   -------------------------- ------------- 5.2/7.9 MB 3.5 MB/s eta 0:00:01
   --------------------------------- ------ 6.6/7.9 MB 3.8 MB/s eta 0:00:01
   ----------------------------------- ---- 7.1/7.9 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------  7.9/7.9 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 7.9/7.9 MB 3.6 MB/s eta 0:00:00

   -------------------- -

In [4]:
#!/usr/bin/env python
# coding: utf-8

import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

# Load the dataset
data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv'
)

# Initialize Dash app
app = dash.Dash(__name__)

# List of years
year_list = [i for i in range(1980, 2024)]

# App layout
app.layout = html.Div([
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={'textAlign': 'center', 'color': '#503D36', 'fontSize': 24}
    ),
    
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=[
                {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
                {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
            ],
            value='Yearly Statistics',
            placeholder='Select a report type'
        )
    ], style={'width': '50%', 'margin': '10px'}),
    
    html.Div([
        dcc.Dropdown(
            id='select-year',
            options=[{'label': year, 'value': year} for year in year_list],
            value=1980,
            placeholder='Select Year'
        )
    ], style={'width': '50%', 'margin': '10px'}),
    
    html.Div(id='output-container', style={'display': 'flex', 'flexDirection': 'column'})
])

# Callback to enable/disable year dropdown
@app.callback(
    Output('select-year', 'disabled'),
    Input('dropdown-statistics', 'value')
)
def toggle_year_dropdown(selected_statistics):
    return selected_statistics != 'Yearly Statistics'

# Callback to generate plots
@app.callback(
    Output('output-container', 'children'),
    [Input('dropdown-statistics', 'value'),
     Input('select-year', 'value')]
)
def update_output_container(selected_statistics, input_year):
    if selected_statistics == 'Recession Period Statistics':
        recession_data = data[data['Recession'] == 1]
        
        # Plot 1: Average automobile sales per year
        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(figure=px.line(
            yearly_rec, x='Year', y='Automobile_Sales',
            title="Average Automobile Sales Fluctuation over Recession Period",
            labels={'Automobile_Sales': 'Average Sales', 'Year': 'Year'}
        ))

        # Plot 2: Average sales by vehicle type
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        R_chart2 = dcc.Graph(figure=px.bar(
            average_sales, x='Vehicle_Type', y='Automobile_Sales',
            title="Average Vehicles Sold by Type during Recession",
            labels={'Automobile_Sales': 'Average Sales', 'Vehicle_Type': 'Vehicle Type'}
        ))

        # Plot 3: Advertising expenditure share
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(figure=px.pie(
            exp_rec, names='Vehicle_Type', values='Advertising_Expenditure',
            title="Advertising Expenditure Share by Vehicle Type during Recession"
        ))

        # Plot 4: Effect of unemployment rate
        unemp_data = recession_data.groupby(['Vehicle_Type', 'unemployment_rate'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(figure=px.bar(
            unemp_data, x='unemployment_rate', y='Automobile_Sales', color='Vehicle_Type',
            labels={'unemployment_rate': 'Unemployment Rate', 'Automobile_Sales': 'Average Automobile Sales'},
            title='Effect of Unemployment Rate on Vehicle Type and Sales'
        ))

        return [
            html.Div([html.Div(R_chart1), html.Div(R_chart2)], style={'display': 'flex'}),
            html.Div([html.Div(R_chart3), html.Div(R_chart4)], style={'display': 'flex'})
        ]

    elif selected_statistics == 'Yearly Statistics' and input_year:
        yearly_data = data[data['Year'] == int(input_year)]
        
        # Plot 1: Yearly Automobile Sales (line)
        yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        Y_chart1 = dcc.Graph(figure=px.line(
            yas, x='Year', y='Automobile_Sales',
            title='Yearly Automobile Sales (All Years)',
            labels={'Automobile_Sales': 'Average Sales', 'Year': 'Year'}
        ))

        # Plot 2: Monthly Automobile Sales
        mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
        Y_chart2 = dcc.Graph(figure=px.line(
            mas, x='Month', y='Automobile_Sales',
            title=f'Total Monthly Automobile Sales in {input_year}',
            labels={'Automobile_Sales': 'Total Sales', 'Month': 'Month'}
        ))

        # Plot 3: Average vehicles sold by type
        avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        Y_chart3 = dcc.Graph(figure=px.bar(
            avr_vdata, x='Vehicle_Type', y='Automobile_Sales',
            title=f'Average Vehicles Sold by Vehicle Type in {input_year}',
            labels={'Automobile_Sales': 'Average Sales', 'Vehicle_Type': 'Vehicle Type'}
        ))

        # Plot 4: Advertisement Expenditure by type
        exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        Y_chart4 = dcc.Graph(figure=px.pie(
            exp_data, names='Vehicle_Type', values='Advertising_Expenditure',
            title=f'Total Advertisement Expenditure by Vehicle Type in {input_year}'
        ))

        return [
            html.Div([html.Div(Y_chart1), html.Div(Y_chart2)], style={'display': 'flex'}),
            html.Div([html.Div(Y_chart3), html.Div(Y_chart4)], style={'display': 'flex'})
        ]

    return None

# Run the app
if __name__ == '__main__':
    app.run(debug=True)
